# Sandbox — exploración previa

Cuaderno de **pruebas**, no de presentación. Aquí es donde se descubrieron los
tres problemas que luego condicionaron el diseño del pipeline:

1. `yfinance` ya no devuelve `Adj Close` por defecto.
2. La descarga en paralelo falla silenciosamente por bloqueo de la caché SQLite.
3. Madrid y Nueva York no comparten calendario de festivos.

La lógica definitiva vive en `src/utils/`; esto es el banco de trabajo.

In [ ]:
import sys, warnings
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))   # .../src
warnings.simplefilter("ignore")

import numpy as np
import pandas as pd
import yfinance as yf

from utils import config as cfg

print("pandas", pd.__version__, "| yfinance", yf.__version__)

## 1. ¿Qué devuelve exactamente `yf.download`?

Primera sorpresa: la forma de las columnas. No es un DataFrame plano, es un
**MultiIndex** cuyo orden ha cambiado entre versiones de la librería.

In [ ]:
muestra = yf.download(
    cfg.TICKERS, start="2024-01-01", end="2024-02-01",
    auto_adjust=False, actions=False, group_by="column",
    progress=False, threads=False,
)
print("Shape:", muestra.shape)
print("Niveles del MultiIndex:")
print("  nivel 0 (campo) :", sorted(set(muestra.columns.get_level_values(0))))
print("  nivel 1 (ticker):", sorted(set(muestra.columns.get_level_values(1))))

Shape: (22, 18)
Niveles del MultiIndex:
  nivel 0 (campo) : ['Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume']
  nivel 1 (ticker): ['ITX.MC', 'PDD', '^IBEX']


### Problema 1 — `auto_adjust` por defecto

Desde `yfinance` 0.2.51 el valor por defecto es `auto_adjust=True`, que
**elimina** la columna `Adj Close`. Hay que pedirla explícitamente.

Esto importa de verdad: Inditex reparte dividendo ordinario + extraordinario
cada año. Veamos la diferencia entre el cierre bruto y el ajustado.

In [ ]:
comparacion = pd.DataFrame({
    "Close": muestra["Close"]["ITX.MC"],
    "Adj Close": muestra["Adj Close"]["ITX.MC"],
})
comparacion["diferencia_%"] = (
    100 * (comparacion["Close"] / comparacion["Adj Close"] - 1)
)
comparacion.head()

,Close,Adj Close,diferencia_%
Date,,,
2024-01-02,39.150002,35.470383,10.373778
2024-01-03,37.950001,34.383163,10.373790
2024-01-04,38.009998,34.437523,10.373787
2024-01-05,37.619999,34.084179,10.373787
2024-01-08,38.070000,34.491886,10.373783


La diferencia es sistemática (~10% en este periodo): son los dividendos
acumulados. Si calculásemos rentabilidades sobre `Close`, cada fecha
ex-dividendo aparecería como un desplome que **nunca ocurrió**.

→ Decisión que llega a `ingestion.py`: `auto_adjust=False` + *fallback* a
`Close` con aviso explícito si una versión futura retira la columna.

### Problema 2 — la descarga en paralelo puede fallar EN SILENCIO

Con `threads=True`, `yfinance` lanza *workers* que compiten por su caché
SQLite local. En la primera ejecución de este proyecto saltó esto:

```
1 Failed download:
['PDD']: OperationalError('database is locked')
```

Lo grave no es el error: es que `yfinance` **no lanza una excepción**. Devuelve
esa columna entera vacía y el programa continúa. Un pipeline descuidado habría
publicado conclusiones sobre dos tickers creyendo que analizaba tres.

**Es intermitente** — depende de si la caché está caliente y del momento — y
eso lo hace peor, no mejor: un fallo que no se reproduce a voluntad es un fallo
que aparecerá el día de la presentación. La celda siguiente compara ambos
modos; puede que hoy coincidan.

In [ ]:
# Comparación de ambos modos. OJO: que hoy coincidan no significa que
# threads=True sea seguro - el bloqueo de la caché es intermitente.
paralelo = yf.download(
    cfg.TICKERS, start="2024-01-01", end="2024-02-01",
    auto_adjust=False, actions=False, group_by="column",
    progress=False, threads=True,
)

comparativa = pd.DataFrame({
    "NaN_threads_True": paralelo["Adj Close"].isna().sum(),
    "NaN_threads_False": muestra["Adj Close"].isna().sum(),
})
comparativa["columna_vacia_en_paralelo"] = (
    paralelo["Adj Close"].isna().all()
)
print("Un ticker con la columna COMPLETAMENTE vacía = descarga fallida.\n")
comparativa

Un ticker con la columna COMPLETAMENTE vacía = descarga fallida.



,NaN_threads_True,NaN_threads_False,columna_vacia_en_paralelo
Ticker,,,
ITX.MC,0,0,False
PDD,1,1,False
^IBEX,0,0,False


Sea cual sea el resultado de hoy, la decisión de diseño no cambia: el coste de
serializar tres descargas es de segundos, y el coste de un competidor ausente
es el proyecto entero.

→ Decisión: `threads=False` (son 3 tickers, la velocidad da igual) y, en
`_extract_field`, un `RuntimeError` explícito si falta algún ticker en lugar de
continuar con un universo parcial.

## 2. Problema 3 — los calendarios no coinciden

In [ ]:
panel = yf.download(
    cfg.TICKERS, start=cfg.START_DATE, end=cfg.END_DATE,
    auto_adjust=False, actions=False, group_by="column",
    progress=False, threads=False,
)["Adj Close"][cfg.TICKERS]

print("Sesiones en el calendario unión:", len(panel))
print("\nFestivos (NaN) por ticker:")
print(panel.isna().sum())
print(f"\nFilas que sobrevivirían a un .dropna(): {len(panel.dropna())}")
print(f"Coste del .dropna(): {len(panel) - len(panel.dropna())} filas "
      f"({100 * (1 - len(panel.dropna()) / len(panel)):.1f}% de la muestra)")

Sesiones en el calendario unión: 991

Festivos (NaN) por ticker:
Ticker
ITX.MC     8
PDD       27
^IBEX      8
dtype: int64

Filas que sobrevivirían a un .dropna(): 956
Coste del .dropna(): 35 filas (3.5% de la muestra)


### ¿Qué días concretos se perderían?

Esto es lo que convenció de usar forward-fill: no son datos rotos, son
**festivos nacionales** de dos países distintos.

In [ ]:
perdidos = panel[panel.isna().any(axis=1)].copy()
perdidos["mercado_cerrado"] = [
    ", ".join(panel.columns[fila.isna()]) for _, fila in perdidos.iterrows()
]
perdidos["mes"] = perdidos.index.month
print("Distribución por mes de las sesiones que un .dropna() destruiría:")
print(perdidos["mes"].value_counts().sort_index().to_string())
print()
perdidos[["mercado_cerrado"]].head(12)

Distribución por mes de las sesiones que un .dropna() destruiría:
mes
1     5
2     4
4     4
5     6
6     3
7     4
9     4
11    3
12    2



Ticker,mercado_cerrado
Date,
2021-01-18,PDD
2021-02-15,PDD
2021-04-05,"ITX.MC, ^IBEX"
2021-05-31,PDD
2021-07-05,PDD
2021-09-06,PDD
2021-11-25,PDD
2021-12-31,"ITX.MC, ^IBEX"
2022-01-17,PDD


**El sesgo no es aleatorio.** Los festivos se concentran en meses concretos
y la Semana Santa cae siempre en marzo/abril — justo en el borde de la ventana
BBC que vamos a estudiar. Un `.dropna()` habría recortado precisamente la
temporada que queremos medir.

→ Decisión: `.ffill()` + eliminación únicamente de los `NaN` de frontera.

## 3. Primer vistazo a los datos (¿hay historia aquí?)

In [ ]:
retornos = panel.ffill().pct_change().iloc[1:]
resumen = pd.DataFrame({
    "media_diaria_%": 100 * retornos.mean(),
    "vol_anualizada_%": 100 * retornos.std(ddof=1) * np.sqrt(252),
    "peor_dia_%": 100 * retornos.min(),
    "mejor_dia_%": 100 * retornos.max(),
    "dias_bajo_-5%": (retornos < -0.05).sum(),
})
resumen.round(2)

,media_diaria_%,vol_anualizada_%,peor_dia_%,mejor_dia_%,dias_bajo_-5%
Ticker,,,,,
ITX.MC,0.10,24.92,-6.95,7.74,5
PDD,0.08,75.58,-28.51,56.06,87
^IBEX,0.04,15.88,-4.96,4.88,0


Aquí ya se ve la historia: PDD tiene **3 veces** la volatilidad de Inditex y
decenas de sesiones por debajo del −5%, frente a un puñado en Zara. Eso es lo
que se formaliza como H2 en la memoria.

## 4. ¿Se nota la temporada BBC en el volumen?

In [ ]:
volumen = yf.download(
    cfg.TICKERS, start=cfg.START_DATE, end=cfg.END_DATE,
    auto_adjust=False, actions=False, group_by="column",
    progress=False, threads=False,
)["Volume"][cfg.TICKERS]

por_mes = volumen.groupby(volumen.index.month).mean()
por_mes.index.name = "mes"
indice_estacional = (por_mes / volumen.mean()).round(2)
indice_estacional.columns = [f"{c}_indice" for c in indice_estacional.columns]
indice_estacional

,ITX.MC_indice,PDD_indice,^IBEX_indice
mes,,,
1,0.72,0.82,1.07
2,0.77,0.68,1.22
3,1.08,1.56,1.40
4,1.74,0.87,1.07
5,1.24,1.01,1.03
6,0.84,0.98,0.91
7,0.68,0.82,0.87
8,0.65,1.26,0.67
9,0.84,1.03,0.92


Abril destaca con claridad en `ITX.MC` y **no** en `PDD`. Ese contraste es el
germen de H3.

**Nota:** `^IBEX` también reporta volumen, pero en un índice es un agregado
sintético, no acciones negociadas. Por eso `config.VOLUME_TICKERS` lo excluye
del análisis de liquidez.

---

## Conclusiones de la exploración previa

| Hallazgo | Decisión en el pipeline |
|---|---|
| `auto_adjust=True` elimina `Adj Close` | `auto_adjust=False` + fallback con aviso |
| `threads=True` vacía un ticker en silencio | `threads=False` + `RuntimeError` si falta alguno |
| 35 sesiones con algún mercado cerrado, sesgadas por mes | `.ffill()` y solo `NaN` de frontera eliminados |
| El ffill crea retornos 0% artificiales | Máscara de sesiones reales → H2 se calcula dos veces |
| El volumen de un índice es sintético | `^IBEX` excluido de H3 |

El resto del trabajo está en `src/memoria.ipynb`.